In [1]:
# imports

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import cvxpy as cp

c:\Users\phili\miniconda3\envs\phpo\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
main_df = pd.read_feather("./Data/US_datastream/US_data_panel_filtered_0.15.feather", columns=["Date", "DSCD", "Return"])
main_df = main_df.sort_values(by="Date")

In [3]:
# metrics for stock_selection 

# for now I also want start with just the pseudo Sharpe
def metric_sharpe(ret_matrix: pd.DataFrame):
    return ret_matrix.mean() / ret_matrix.std(ddof=1) # IMO this is a pretty crazy line since it does so much

In [4]:
# helpful functions

# function for quickly checking what certain vars actually are at the moment

def get_info(**kwargs):

    for key,value in kwargs.items():
        
        print(key,"|",value, "|", type(value))


# function for changing the format of the initial DataFrame

def construct_ret_matrix(panel: pd.DataFrame) -> pd.DataFrame:

    return panel.pivot(index="Date", columns="DSCD", values="Return")

# filter for stocks with available return data

def filter_av_returns(panel: pd.DataFrame) -> list:

    total_days = panel["Date"].nunique()

    valid_stocks = []
    for dscd, dscd_panel in panel.groupby("DSCD", sort=False):
        
        dscd_filtered_series = dscd_panel["Return"].dropna()
        #dscd_filtered_series = dscd_filtered_series[dscd_filtered_series != 0]

        if len(dscd_filtered_series) != total_days: # I don't allow even a single missing return (but 0 returns are in for now) -> smaler scope helps me here
            continue
        
        valid_stocks.append(dscd)
    
    return valid_stocks

# my individual stock selection function, should also work variable by giving it different metrics

def select_stocks(ret_matrix: pd.DataFrame, nstocks: int, metric: callable, by_highest: bool) -> list:
    
    scores = metric(ret_matrix)
    return scores.sort_values(ascending=(not by_highest)).head(nstocks).index.tolist()

In [5]:
# TODO dive into detail here, for now just make it work
# TODO e.g. the decision to fill missing returns with 0 is interesting
# TODO e.g. the scaling is interesting 

def sample_cov(R: pd.DataFrame):
    
    Sigma = R.cov().values
    Sigma = Sigma + 1e-4 * np.eye(Sigma.shape[0]) # added this to open quetions, I know what it does but not fully why we do it
    return pd.DataFrame(Sigma, index=R.columns, columns=R.columns)

`sample_cov()`

for stocks i,j compute:

$$\Sigma_{i,j} = \frac{1}{T-1} \sum_t{(r_{i,t} - \bar{r}_i)(r_{j,t} - \bar{r}_j)}$$

In [6]:
def min_var_weights(Sigma: pd.DataFrame):
    
    S = cp.psd_wrap(Sigma.values) # use ndarray instead of DF and trust to be positive semidefinite
    
    X = cp.Variable(S.shape[0]) # x_i for each stock i in S

    constraints = [X >= 0,
                   X <= 1,
                   X.sum() == 1.0]
    
    opti_problem = cp.Problem(cp.Minimize(cp.quad_form(X,S)), constraints=constraints)

    opti_problem.solve()

    # Error handling
    if opti_problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
        raise ValueError(f"min_var optimisation failed. Status: {opti_problem.status}")
    
    return pd.Series(X.value.round(6), index=Sigma.index)

`min_var_weights()`

solve:
$$\min_x{x^T\sum{x}}$$
w.r.t.:
$$\text{no overinvesting}$$
$$\sum_i{x_i}=1$$
$$\text{long-only}$$
$$0 \leq x_i \leq 1$$

In [14]:
# Since it's SUPER important for me to fluently know this, I add it here for my personal understanding

def calc_pf_returns(R: pd.DataFrame, w: pd.Series):

    return R.loc[:, w.index].values @ w.values # cool syntax

`calc_pf_returns`

multiply the returns of each stock for every with the corresponding daily return:

$$Rw$$

$$\text{e.g.}$$

$$R=\begin{bmatrix} 0,01 & 0,02 & -0,01 \\ -0,03 & 0,01 & 0,04 \end{bmatrix}$$

$$w=\begin{bmatrix} 0,5 \\ 0,3 \\ 0,2 \end{bmatrix}$$

$$\text{which gives}$$

$$r_{t_1}=0,01*0,5+0,02*0,3+(-0,01)*0,2=0,009$$

$$r_{t_2}=(-0,03)*0,5+0,01*0,3+0,04*0,2=-0,004$$

In [12]:
# everything regardings the simulated investors

# my individual investor class

class Investor:

    def __init__(self, strategy: str, cov_estimator: str):
        
        # attributes

        self.strategy = strategy
        self.cov_estimator = cov_estimator
        self.weight_history = []
        self.return_history = []

        if self.strategy == "TODO":
            pass
        else: # default strategy = min_var
            self.weights = min_var_weights
        if self.cov_estimator == "TODO":
            pass
        else: # default covariance estimation is in-sample 
            self.covar = sample_cov
    
    # heart piece of the whole thing
    def rebalance(self, sel_R: pd.DataFrame, period: np.ndarray, stocks: list):

        # Build Portfolio

        S = self.covar(R=sel_R)
        w = self.weights(Sigma=S).sort_index()
        
        sorted_dscds = w.index.tolist()

        # Hold Portfolio

        panel_held = df[(df["Date"].isin(period)) & (df["DSCD"].isin(stocks))]
        
        R_held = construct_ret_matrix(panel=panel_held).reindex(index=period, columns=sorted_dscds) # no fillna() currently

        self.add_pf_returns(ret_mat=R_held, weights=w)

    def add_pf_returns(self, ret_mat: pd.DataFrame, weights: pd.Series):
        
        R = ret_mat
        w = weights

        pf_returns = calc_pf_returns(R=R, w=w) 

        self.weight_history.append(w)
        self.return_history.append(pd.Series(pf_returns, index=R.index))

In [15]:
df = main_df[["Date", "Return","DSCD"]]

un_dates = np.sort(df["Date"].unique())
T = len(un_dates)
un_dates_cut = un_dates[2000:6000] # just to scale down a bit for this experimental scope
T_cut = len(un_dates_cut)

estimation_window = 512
holding_period = 21

times_to_rebalance = range(estimation_window, T, holding_period)

inv = Investor(strategy="Minimum Variance", cov_estimator="Sample")

for t in tqdm(times_to_rebalance):

    est_start = un_dates_cut[t - estimation_window]
    est_end = un_dates_cut[t - 1]

    prevent_overflow_idx = min(t + holding_period, T_cut)

    holding_period = un_dates_cut[t:prevent_overflow_idx]

    estimation_sector = df[(df["Date"] >= est_start) & (df["Date"] <= est_end)]

    filtered_stocks = filter_av_returns(estimation_sector)
    filtered_sector = estimation_sector[estimation_sector["DSCD"].isin(filtered_stocks)]
    filtered_sector_R = construct_ret_matrix(panel=filtered_sector)

    selected_stocks = select_stocks(ret_matrix=filtered_sector_R, nstocks=10, metric=metric_sharpe, by_highest=True)
    selected_sector = filtered_sector[filtered_sector["DSCD"].isin(selected_stocks)]
    selected_sector_R = filtered_sector_R.loc[:, selected_stocks] # should be the same as just caling construct_ret_matrix again

    selected_sector_R = selected_sector_R.loc[:, selected_sector_R.columns.sort_values()] # not sure why we do this exavtly, I think because rows should match for our matrix calculations later -> added it to open questions

    inv.rebalance(sel_R=selected_sector_R, period=holding_period, stocks=selected_stocks)

    print(inv.return_history)

    break

  0%|          | 0/363 [00:03<?, ?it/s]

[Date
2003-09-22   -0.007035
2003-09-23    0.006566
2003-09-24   -0.003110
2003-09-25   -0.006654
2003-09-26    0.001082
2003-09-29   -0.000065
2003-09-30    0.004973
2003-10-01    0.004707
2003-10-02    0.005106
2003-10-03    0.009500
2003-10-06    0.012197
2003-10-07    0.001620
2003-10-08    0.001679
2003-10-09    0.007606
2003-10-10    0.011745
2003-10-13    0.008534
2003-10-14    0.005510
2003-10-15   -0.002489
2003-10-16    0.000720
2003-10-17   -0.008883
2003-10-20   -0.007171
dtype: float64]
